In [1]:
from sklearn.svm import SVC, SVR, NuSVR, NuSVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.kernel_ridge import KernelRidge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, brier_score_loss

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
import gc

import xgboost as xgb

import statsmodels.api as sm
import tqdm
from sklearn.metrics import roc_auc_score

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import catboost as cb
from collections import defaultdict

warnings.filterwarnings("ignore")
pd.set_option("display.max_column", 999)
data_dir = '/kaggle/input/competitions/march-machine-learning-mania-2026'

# W -> women's basketball, M -> men's basketball
M_regular_results = pd.read_csv(f"{data_dir}/MRegularSeasonDetailedResults.csv")
M_tourney_results = pd.read_csv(f"{data_dir}/MNCAATourneyDetailedResults.csv")
M_seeds = pd.read_csv(f"{data_dir}/MNCAATourneySeeds.csv")

W_regular_results = pd.read_csv(f"{data_dir}/WRegularSeasonDetailedResults.csv")
W_tourney_results = pd.read_csv(f"{data_dir}/WNCAATourneyDetailedResults.csv")
W_seeds = pd.read_csv(f"{data_dir}/WNCAATourneySeeds.csv")

In [2]:
# join men's and women's data into one file
regular_results = pd.concat([M_regular_results, W_regular_results])
tourney_results = pd.concat([M_tourney_results, W_tourney_results])
seeds = pd.concat([M_seeds, W_seeds])

In [3]:
season = 2000  # change if you want different cutoff year for your models
regular_results = regular_results.loc[regular_results["Season"] >= season]
tourney_results = tourney_results.loc[tourney_results["Season"] >= season]
seeds = seeds.loc[seeds["Season"] >= season]

In [4]:
wloc = {'H':1, 'A':-1, 'N': np.nan} #{'H':1, 'A':0, 'N': 0}  # {'H':1, 'A':-1, 'N': 0} #np.nan} #-1}
regular_results['WHome'] = regular_results['WLoc'].map(lambda x: wloc[x])
regular_results['WHome'].value_counts()

WHome
 1.0    121160
-1.0     69839
Name: count, dtype: int64

In [5]:
def build_base_features(df):
    """
    Cleans and doubles the Kaggle NCAA box score dataset, creating a symmetrical 
    T1 vs T2 format with normalized overtime stats.
    """
    # Make a copy so we don't accidentally modify the original dataframe in memory
    df = df.copy()
    
    # ---------------------------------------------------------
    # 1. HOME/AWAY MAPPING
    # ---------------------------------------------------------
    # H = 1 (Home), A = -1 (Away), N = 0 (Neutral)
    # Using 0 for neutral is usually better for machine learning than NaN
    loc_map = {'H': 1, 'A': -1, 'N': 0}
    df['WLoc_Num'] = df['WLoc'].map(loc_map).fillna(0)
    df['LLoc_Num'] = -df['WLoc_Num'] # The loser has the exact opposite location status
    
    # ---------------------------------------------------------
    # 2. OVERTIME NORMALIZATION
    # ---------------------------------------------------------
    # Standard game is 40 minutes. Each OT adds 5 minutes.
    # We calculate the ratio to scale counting stats down to a 40-minute equivalent.
    minutes_played = 40 + (df['NumOT'] * 5)
    ot_modifier = 40 / minutes_played
    
    # All the counting stats we want to normalize
    stat_cols = [
        'Score', 'FGM', 'FGA', 'FGM3', 'FGA3', 'FTM', 'FTA', 
        'OR', 'DR', 'Ast', 'TO', 'Stl', 'Blk', 'PF'
    ]
    
    for col in stat_cols:
        df[f'W{col}'] = df[f'W{col}'] * ot_modifier
        df[f'L{col}'] = df[f'L{col}'] * ot_modifier

    # ---------------------------------------------------------
    # 3. DATASET DOUBLING (THE PERSPECTIVE SWAP)
    # ---------------------------------------------------------
    # Create the Winner's perspective (Team 1 Wins)
    df_winner = pd.DataFrame()
    df_winner['Season'] = df['Season']
    df_winner['DayNum'] = df['DayNum']
    df_winner['T1_TeamID'] = df['WTeamID']
    df_winner['T2_TeamID'] = df['LTeamID']
    df_winner['T1_Loc'] = df['WLoc_Num']
    
    for col in stat_cols:
        df_winner[f'T1_{col}'] = df[f'W{col}']
        df_winner[f'T2_{col}'] = df[f'L{col}']
        
    df_winner['Target_Win'] = 1 # Team 1 won this matchup
    
    # Create the Loser's perspective (Team 1 Loses)
    df_loser = pd.DataFrame()
    df_loser['Season'] = df['Season']
    df_loser['DayNum'] = df['DayNum']
    df_loser['T1_TeamID'] = df['LTeamID']
    df_loser['T2_TeamID'] = df['WTeamID']
    df_loser['T1_Loc'] = df['LLoc_Num']
    
    for col in stat_cols:
        df_loser[f'T1_{col}'] = df[f'L{col}']
        df_loser[f'T2_{col}'] = df[f'W{col}']
        
    df_loser['Target_Win'] = 0 # Team 1 lost this matchup
    
    # Smash them together into one master dataset
    combined_df = pd.concat([df_winner, df_loser], ignore_index=True)
    
    # ---------------------------------------------------------
    # 4. GENDER LABELING
    # ---------------------------------------------------------
    # Men's Team IDs start with 1 (e.g., 1000s). Women's start with 3 (e.g., 3000s).
    combined_df['Is_Mens_Tourney'] = (combined_df['T1_TeamID'] < 2000).astype(int)

    return combined_df

# Usage:
regular_data = build_base_features(regular_results)
regular_data.head()

,Season,DayNum,T1_TeamID,T2_TeamID,T1_Loc,T1_Score,T2_Score,T1_FGM,T2_FGM,T1_FGA,T2_FGA,T1_FGM3,T2_FGM3,T1_FGA3,T2_FGA3,T1_FTM,T2_FTM,T1_FTA,T2_FTA,T1_OR,T2_OR,T1_DR,T2_DR,T1_Ast,T2_Ast,T1_TO,T2_TO,T1_Stl,T2_Stl,T1_Blk,T2_Blk,T1_PF,T2_PF,Target_Win,Is_Mens_Tourney
0,2003,10,1104,1328,0,68.0,62.0,27.0,22.0,58.0,53.0,3.0,2.0,14.0,10.0,11.0,16.0,18.0,22.0,14.0,10.0,24.0,22.0,13.0,8.0,23.0,18.0,7.0,9.0,1.0,2.0,22.0,20.0,1,1
1,2003,10,1272,1393,0,70.0,63.0,26.0,24.0,62.0,67.0,8.0,6.0,20.0,24.0,10.0,9.0,19.0,20.0,15.0,20.0,28.0,25.0,16.0,7.0,13.0,12.0,4.0,8.0,4.0,6.0,18.0,16.0,1,1
2,2003,11,1266,1437,0,73.0,61.0,24.0,22.0,58.0,73.0,8.0,3.0,18.0,26.0,17.0,14.0,29.0,23.0,17.0,31.0,26.0,22.0,15.0,9.0,10.0,12.0,5.0,2.0,2.0,5.0,25.0,23.0,1,1
3,2003,11,1296,1457,0,56.0,50.0,18.0,18.0,38.0,49.0,3.0,6.0,9.0,22.0,17.0,8.0,31.0,15.0,6.0,17.0,19.0,20.0,11.0,9.0,12.0,19.0,14.0,4.0,2.0,3.0,18.0,23.0,1,1
4,2003,11,1400,1208,0,77.0,71.0,30.0,24.0,61.0,62.0,6.0,6.0,14.0,16.0,11.0,17.0,13.0,27.0,17.0,21.0,22.0,15.0,12.0,12.0,14.0,10.0,4.0,7.0,4.0,1.0,20.0,14.0,1,1


In [6]:
def get_season_averages(regular_df):
    """
    Takes the doubled regular season data and calculates season-long 
    Offensive (Avg) and Defensive (Avg_Opp) stats for every team.
    """
    # The stats we care about
    stat_cols = ['Score', 'FGM', 'FGA', 'FGM3', 'FGA3', 'FTM', 'FTA', 
                 'OR', 'DR', 'Ast', 'TO', 'Stl', 'Blk', 'PF']
    
    # Grab both T1 (Offense) and T2 (Defense) columns
    cols_to_avg = [f'T1_{col}' for col in stat_cols] + [f'T2_{col}' for col in stat_cols]
    
    # Group by Season and Team
    season_avgs = regular_df.groupby(['Season', 'Is_Mens_Tourney', 'T1_TeamID'])[cols_to_avg].mean().reset_index()
    
    # Rename columns so they make sense
    rename_dict = {'T1_TeamID': 'TeamID'}
    for col in stat_cols:
        rename_dict[f'T1_{col}'] = f'Avg_{col}'
        rename_dict[f'T2_{col}'] = f'Avg_Opp_{col}' # What they allowed opponents to do
        
    return season_avgs.rename(columns=rename_dict)

#season_averages = get_season_averages(regular_data)


In [7]:
#Join with regular data
#season_averages

In [8]:
# extract seed number from `Seed` field
seeds["seed"] = seeds["Seed"].apply(lambda x: int(x[1:3]))
seeds['Is_Mens_Tourney'] = (seeds['TeamID'] < 2000).astype(int)
seeds = seeds.drop(columns=['Seed'])
seeds

,Season,TeamID,seed,Is_Mens_Tourney
960,2000,1181,1,1
961,2000,1396,2,1
962,2000,1329,3,1
963,2000,1228,4,1
964,2000,1196,5,1
...,...,...,...,...
1807,2026,3211,12,0
1808,2026,3453,13,0
1809,2026,3158,14,0
1810,2026,3239,15,0


In [9]:
seeds_T1 = seeds[["Season", "TeamID", "seed"]].copy()
seeds_T2 = seeds[["Season", "TeamID", "seed"]].copy()
seeds_T1.columns = ["Season", "T1_TeamID", "T1_seed"]
seeds_T2.columns = ["Season", "T2_TeamID", "T2_seed"]

In [10]:
def calculate_end_of_season_elo(doubled_df, base_elo=1500, k_factor=20, hca=100):
    """
    Calculates End-of-Season Elo ratings directly from a perspective-doubled dataset.
    Returns a clean DataFrame ready to be merged onto your 2026 Tournament Test Set.
    """
    # 1. UNDO THE DOUBLING
    # By filtering for Target_Win == 1, we get exactly one row per game 
    # where T1 is ALWAYS the winner and T2 is ALWAYS the loser.
    unique_games = doubled_df[doubled_df['Target_Win'] == 1].copy()
    
    # 2. Sort chronologically to prevent data leakage (time travel)
    unique_games = unique_games.sort_values(by=['Season', 'DayNum'])
    
    final_elos = []

    # 3. Group by both Season AND Gender so Men's and Women's ratings stay completely separate
    for (season, is_mens), season_df in unique_games.groupby(['Season', 'Is_Mens_Tourney']):
        
        # Hard reset all teams in this specific bracket to the base_elo
        teams = set(season_df['T1_TeamID']) | set(season_df['T2_TeamID'])
        elo_dict = {team: base_elo for team in teams}
        
        # 4. Iterate chronologically through the season
        for idx, row in season_df.iterrows():
            w_team = row['T1_TeamID']  # T1 is always the winner in this filtered subset
            l_team = row['T2_TeamID']  # T2 is always the loser
            w_score = row['T1_Score']
            l_score = row['T2_Score']
            w_loc = row['T1_Loc']      # 1 = Home, -1 = Away, 0 = Neutral
            
            # Fetch Current Elos
            w_elo = elo_dict[w_team]
            l_elo = elo_dict[l_team]
            
            # --- 538 MATH ---
            # Apply Home Court Advantage
            # If w_loc == 1, Winner was at Home. If w_loc == -1, Loser was at Home.
            w_elo_adj = w_elo + (hca if w_loc == 1 else 0)
            l_elo_adj = l_elo + (hca if w_loc == -1 else 0)
            
            # Calculate Expected Win Probability
            expected_win = 1.0 / (1.0 + 10.0 ** ((l_elo_adj - w_elo_adj) / 400.0))
            
            # Calculate Margin of Victory (MoV) Multiplier
            mov = w_score - l_score
            elo_diff = w_elo - l_elo 
            
            mov_multiplier = np.log(mov + 1.0) * (2.2 / ((elo_diff * 0.001) + 2.2))
            
            # Floor the multiplier to prevent weird math bounds on massive upsets
            if mov_multiplier < 0.1: 
                mov_multiplier = 0.1
                
            # Calculate Final Elo Change
            change = k_factor * mov_multiplier * (1.0 - expected_win)
            
            # Update Dictionary
            elo_dict[w_team] = w_elo + change
            elo_dict[l_team] = l_elo - change

        # 5. End of season: Save the final snapshot for the 2026 Tournament predictions
        for team in teams:
            final_elos.append({
                'Season': season,
                'Is_Mens_Tourney': is_mens,
                'TeamID': team,
                'End_Season_Elo': elo_dict[team]
            })

    return pd.DataFrame(final_elos)

# --- HOW TO RUN IT ---
# Just pass your entire dataset (with the 0s and 1s) directly into the function:
#final_elo_df = calculate_end_of_season_elo(regular_data, base_elo=1500)
#final_elo_df

In [11]:
def get_advanced_season_stats(regular_df):
    """
    Calculates KenPom-style Offensive and Defensive Efficiency (Per 100 Possessions)
    to create a native Net Rating for every team in the dataset.
    """
    df = regular_df.copy()
    
    # 1. Calculate Estimated Possessions for the game
    # Standard basketball formula: FGA - OR + TO + (0.475 * FTA)
    df['T1_Possessions'] = df['T1_FGA'] - df['T1_OR'] + df['T1_TO'] + (0.475 * df['T1_FTA'])
    df['T2_Possessions'] = df['T2_FGA'] - df['T2_OR'] + df['T2_TO'] + (0.475 * df['T2_FTA'])
    
    # Average the two teams' possession counts to smooth out rebounding attribution errors
    df['Game_Possessions'] = (df['T1_Possessions'] + df['T2_Possessions']) / 2
    
    # 2. Calculate Game-Level Efficiencies
    # Offense: Points scored per 100 possessions
    df['T1_Off_Eff'] = (df['T1_Score'] / df['Game_Possessions']) * 100
    df['T2_Off_Eff'] = (df['T2_Score'] / df['Game_Possessions']) * 100
    
    # Defense: Points ALLOWED per 100 possessions
    df['T1_Def_Eff'] = df['T2_Off_Eff']
    df['T2_Def_Eff'] = df['T1_Off_Eff']
    
    # 3. Aggregate into Season Averages
    team_stats = []
    grouped = df.groupby(['Season', 'Is_Mens_Tourney', 'T1_TeamID'])
    
    for (season, is_mens, team_id), group in grouped:
        avg_off_eff = group['T1_Off_Eff'].mean()
        avg_def_eff = group['T1_Def_Eff'].mean()
        
        # NET RATING: The ultimate "KenPom" power metric
        net_rating = avg_off_eff - avg_def_eff
        
        team_stats.append({
            'Season': season,
            'Is_Mens_Tourney': is_mens,
            'TeamID': team_id,
            'Avg_Off_Eff': avg_off_eff,
            'Avg_Def_Eff': avg_def_eff,
            'Net_Rating': net_rating,
            # We also keep raw win % as a baseline feature
            'Win_Pct': group['Target_Win'].mean() 
        })
        
    return pd.DataFrame(team_stats)

# Usage in your pipeline:
#advanced_season_stats = get_advanced_season_stats(regular_data)
#advanced_season_stats[advanced_season_stats['Is_Mens_Tourney']==0]

In [12]:
def calculate_win_percentages(doubled_df):
    """
    Calculates Overall, Home, Away, and Neutral win percentages for each team.
    Designed for the perspective-doubled dataset where T1_Loc: 1=Home, -1=Away, 0=Neutral.
    """
    
    # 1. Overall Win Percentage
    overall = doubled_df.groupby(['Season', 'Is_Mens_Tourney', 'T1_TeamID'])['Target_Win'].agg(
        Overall_Win_Pct='mean'
    ).reset_index()
    
    # 2. Home Win Percentage (T1_Loc == 1)
    home_games = doubled_df[doubled_df['T1_Loc'] == 1]
    home = home_games.groupby(['Season', 'Is_Mens_Tourney', 'T1_TeamID'])['Target_Win'].agg(
        Home_Win_Pct='mean'
    ).reset_index()
    
    # 3. Away Win Percentage (T1_Loc == -1)
    away_games = doubled_df[doubled_df['T1_Loc'] == -1]
    away = away_games.groupby(['Season', 'Is_Mens_Tourney', 'T1_TeamID'])['Target_Win'].agg(
        Away_Win_Pct='mean'
    ).reset_index()
    
    # 5. Merge them all together into one clean DataFrame
    win_pct_df = overall.copy()
    win_pct_df = win_pct_df.merge(home, on=['Season', 'Is_Mens_Tourney', 'T1_TeamID'], how='left')
    win_pct_df = win_pct_df.merge(away, on=['Season', 'Is_Mens_Tourney', 'T1_TeamID'], how='left')
    
    # Rename for the final merge later
    win_pct_df = win_pct_df.rename(columns={'T1_TeamID': 'TeamID'})
    
    # 6. SMART IMPUTATION (Crucial for ML)
    # Many low-major teams play ZERO neutral site games during the regular season.
    # If we leave this as NaN or fill it with 0.0, the model will think they are awful on neutral courts.
    # The smartest mathematical fallback is to fill missing location stats with their Overall_Win_Pct.
    win_pct_df['Home_Win_Pct'] = win_pct_df['Home_Win_Pct'].fillna(win_pct_df['Overall_Win_Pct'])
    win_pct_df['Away_Win_Pct'] = win_pct_df['Away_Win_Pct'].fillna(win_pct_df['Overall_Win_Pct'])
    
    return win_pct_df

# --- HOW TO RUN IT ---
#win_percentages_df = calculate_win_percentages(regular_data)
#win_percentages_df

In [13]:
def add_seeds_to_matchups(matchup_df, seeds_df, unseeded_val=17):
    """
    Merges tournament seeds onto T1 and T2, imputes missing seeds, 
    and calculates the Seed Difference.
    """
    df = matchup_df.copy()
    
    # Merge Seed onto T1
    df = pd.merge(df, seeds_df, 
                  left_on=['Season', 'Is_Mens_Tourney', 'T1_TeamID'],
                  right_on=['Season', 'Is_Mens_Tourney', 'TeamID'],
                  how='left').drop(columns=['TeamID'])
    
    df = df.rename(columns={'Seed_Num': 'T1_Seed'})
    
    # Merge Seed onto T2
    df = pd.merge(df, seeds_df, 
                  left_on=['Season', 'Is_Mens_Tourney', 'T2_TeamID'],
                  right_on=['Season', 'Is_Mens_Tourney', 'TeamID'],
                  how='left').drop(columns=['TeamID'])
    
    df = df.rename(columns={'Seed_Num': 'T2_Seed'})
    
    # Impute missing seeds for teams that didn't make the tournament
    df['T1_Seed'] = df['T1_Seed'].fillna(unseeded_val)
    df['T2_Seed'] = df['T2_Seed'].fillna(unseeded_val)
    
    # Calculate the Difference
    # Note: A NEGATIVE difference means T1 is the BETTER team (e.g. 1 seed vs 16 seed = -15)
    df['Seed_Diff'] = df['T1_Seed'] - df['T2_Seed']
    
    return df

In [14]:
# 1. Run your feature generation functions (assuming you've loaded regular_data)
df_efficiency = get_advanced_season_stats(regular_data)
df_elo = calculate_end_of_season_elo(regular_data, base_elo=1500)
df_win_pct = calculate_win_percentages(regular_data)
season_averages = get_season_averages(regular_data)


# 2. Merge them into the Master Team Stats table
master_stats = pd.merge(df_efficiency, df_elo, on=['Season', 'Is_Mens_Tourney', 'TeamID'], how='inner')
master_stats = pd.merge(master_stats, df_win_pct, on=['Season', 'Is_Mens_Tourney', 'TeamID'], how='inner')
master_stats = pd.merge(master_stats, season_averages, on=['Season', 'Is_Mens_Tourney', 'TeamID'], how='inner')
# 1. Merge the Seed_Num onto your master_stats
master_stats = pd.merge(master_stats, seeds[['Season', 'Is_Mens_Tourney', 'TeamID', 'seed']], 
                        on=['Season', 'Is_Mens_Tourney', 'TeamID'], 
                        how='left')

# 2. Impute the unseeded teams (Teams that didn't make it get a 17)
master_stats['seed'] = master_stats['seed'].fillna(17)
master_stats

,Season,Is_Mens_Tourney,TeamID,Avg_Off_Eff,Avg_Def_Eff,Net_Rating,Win_Pct,End_Season_Elo,Overall_Win_Pct,Home_Win_Pct,Away_Win_Pct,Avg_Score,Avg_FGM,Avg_FGA,Avg_FGM3,Avg_FGA3,Avg_FTM,Avg_FTA,Avg_OR,Avg_DR,Avg_Ast,Avg_TO,Avg_Stl,Avg_Blk,Avg_PF,Avg_Opp_Score,Avg_Opp_FGM,Avg_Opp_FGA,Avg_Opp_FGM3,Avg_Opp_FGA3,Avg_Opp_FTM,Avg_Opp_FTA,Avg_Opp_OR,Avg_Opp_DR,Avg_Opp_Ast,Avg_Opp_TO,Avg_Opp_Stl,Avg_Opp_Blk,Avg_Opp_PF,seed
0,2003,1,1102,103.835749,103.601436,0.234313,0.428571,1443.877630,0.428571,0.692308,0.230769,57.250000,19.142857,39.785714,7.821429,20.821429,11.142857,17.107143,4.178571,16.821429,13.000000,11.428571,5.964286,1.785714,18.750000,57.000000,19.285714,42.428571,4.750000,12.428571,13.678571,19.250000,9.607143,20.142857,9.142857,12.964286,5.428571,1.571429,18.357143,17.0
1,2003,1,1103,110.667469,110.408760,0.258709,0.481481,1483.586415,0.481481,0.642857,0.307692,75.967901,26.268313,54.051029,5.296296,15.611523,18.134979,24.580247,9.497119,19.255967,14.723457,12.287243,6.886420,2.297119,19.027160,75.259259,26.794239,55.008230,6.425514,17.741564,15.245267,21.069136,11.443621,21.232922,14.907819,14.753909,6.218930,2.750617,21.566255,17.0
2,2003,1,1104,103.557697,97.794103,5.763594,0.607143,1571.793690,0.607143,0.866667,0.111111,69.015873,23.940476,56.964286,6.329365,19.805556,14.805556,20.849206,13.523810,23.853175,12.059524,13.234127,6.571429,3.777778,17.972222,64.753968,23.150794,55.297619,6.345238,19.067460,12.107143,17.095238,10.857143,22.551587,11.638889,13.789683,5.503968,3.170635,19.170635,10.0
3,2003,1,1105,93.616492,100.207433,-6.590941,0.269231,1322.587296,0.269231,0.416667,0.142857,70.400855,23.938462,60.519658,7.447009,20.373504,15.076923,21.348718,13.252137,22.625641,14.269231,18.272650,9.128205,2.033333,19.789744,75.244444,26.548718,57.847009,6.123932,17.210256,16.023077,23.894872,12.901709,25.925641,15.561538,18.485470,9.217949,4.138462,18.683761,17.0
4,2003,1,1106,93.773763,94.224204,-0.450441,0.464286,1423.252024,0.464286,0.666667,0.357143,63.281746,23.309524,55.019841,6.083333,17.567460,10.579365,16.384921,12.234127,23.761905,11.619048,16.984127,8.305556,3.126984,18.115079,63.444444,21.607143,53.146825,4.761905,15.146825,15.468254,21.876984,11.273810,22.273810,11.722222,15.003968,8.773810,3.154762,16.079365,17.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14306,2026,1,1477,97.212708,109.321507,-12.108799,0.300000,1325.058757,0.300000,0.333333,0.235294,67.066667,23.433333,55.800000,8.066667,25.800000,12.133333,17.866667,7.400000,21.766667,14.633333,12.566667,6.733333,3.133333,16.900000,76.033333,26.833333,58.600000,8.300000,22.266667,14.066667,20.200000,9.966667,24.566667,15.033333,11.033333,7.333333,3.200000,15.100000,17.0
14307,2026,1,1478,104.087180,105.684311,-1.597131,0.451613,1390.523303,0.451613,0.666667,0.294118,72.322581,24.709677,54.129032,7.580645,22.096774,15.322581,21.354839,7.000000,24.903226,13.967742,12.451613,5.483871,3.290323,17.483871,73.903226,25.387097,60.258065,8.612903,23.903226,14.516129,19.677419,9.548387,23.516129,13.935484,9.709677,7.677419,3.032258,17.774194,17.0
14308,2026,1,1479,102.035297,103.044083,-1.008786,0.468750,1479.852029,0.468750,0.769231,0.263158,67.173611,25.368056,56.968750,5.701389,17.753472,10.736111,14.482639,7.267361,21.131944,13.743056,8.506944,6.854167,3.774306,18.079861,67.593750,23.263889,52.013889,5.649306,17.513889,15.416667,22.138889,7.996528,23.451389,12.211806,11.611111,4.993056,2.461806,15.177083,17.0
14309,2026,1,1480,104.089018,111.388414,-7.299396,0.433333,1399.690506,0.433333,0.615385,0.266667,74.417037,27.405185,62.075556,6.791111,20.494815,12.815556,18.131111,9.879259,22.796296,12.137037,10.625185,6.280741,4.608148,17.037037,79.606667,28.082222,59.366667,7.347407,22.183704,16.094815,21.895556,7.898519,23.834074,14.162963,9.722963,6.723704,2.325185,16.769630,17.0


In [15]:
def get_recent_win_pct(regular_df, days=14):
    """
    Calculates the win percentage for each team in their final `days` of the regular season.
    By Kaggle standards, the regular season ends on Day 132.
    """
    df = regular_df.copy()
    
    # Calculate the cutoff day (e.g., 132 - 14 + 1 = 119)
    # Using the max DayNum in the dataset dynamically ensures it works across all seasons
    max_day = df['DayNum'].max()
    cutoff_day = max_day - days + 1
    
    # Filter for only the recent games
    recent_games = df[df['DayNum'] >= cutoff_day]
    
    # Calculate the win percentage in these recent games
    recent_stats = recent_games.groupby(['Season', 'Is_Mens_Tourney', 'T1_TeamID'])['Target_Win'].agg(
        Last_14_Days_Win_Pct='mean'
    ).reset_index()
    
    # Rename column to match your master_stats format
    recent_stats = recent_stats.rename(columns={'T1_TeamID': 'TeamID'})
    
    return recent_stats

In [16]:
# 1. Generate the recent stats
df_recent_win = get_recent_win_pct(regular_data, days=14)

In [17]:
# 2. Merge it into your existing master_stats table
master_stats = pd.merge(master_stats, df_recent_win, on=['Season', 'Is_Mens_Tourney', 'TeamID'], how='left')

# 3. Smart Imputation for teams that didn't play in the last 14 days
master_stats['Last_14_Days_Win_Pct'] = master_stats['Last_14_Days_Win_Pct'].fillna(master_stats['Overall_Win_Pct'])

# (Optional) Verify it worked
master_stats[['Season', 'TeamID', 'Overall_Win_Pct', 'Last_14_Days_Win_Pct']].head()

,Season,TeamID,Overall_Win_Pct,Last_14_Days_Win_Pct
0,2003,1102,0.428571,0.333333
1,2003,1103,0.481481,0.333333
2,2003,1104,0.607143,0.333333
3,2003,1105,0.269231,0.000000
4,2003,1106,0.464286,0.500000


In [18]:
def map_features_to_matchups(matchup_df, stats_df):
    """
    Takes a dataframe of matchups (needs Season, Is_Mens_Tourney, T1_TeamID, T2_TeamID)
    and merges the master stats onto both teams, calculating the differentials.
    """
    df = matchup_df.copy()
    
    # Merge Stats for T1
    df = pd.merge(df, stats_df, 
                  left_on=['Season', 'Is_Mens_Tourney', 'T1_TeamID'],
                  right_on=['Season', 'Is_Mens_Tourney', 'TeamID'],
                  how='left').drop(columns=['TeamID'])
    
    # Rename T1 columns
    rename_dict_t1 = {col: f'T1_{col}' for col in stats_df.columns if col not in ['Season', 'Is_Mens_Tourney', 'TeamID']}
    df = df.rename(columns=rename_dict_t1)
    
    # Merge Stats for T2
    df = pd.merge(df, stats_df, 
                  left_on=['Season', 'Is_Mens_Tourney', 'T2_TeamID'],
                  right_on=['Season', 'Is_Mens_Tourney', 'TeamID'],
                  how='left').drop(columns=['TeamID'])
    
    # Rename T2 columns
    rename_dict_t2 = {col: f'T2_{col}' for col in stats_df.columns if col not in ['Season', 'Is_Mens_Tourney', 'TeamID']}
    df = df.rename(columns=rename_dict_t2)
    
    # Calculate Differentials (T1 - T2)
    features_to_diff = [col for col in stats_df.columns if col not in ['Season', 'Is_Mens_Tourney', 'TeamID', 'Win_Pct']]
    
    for feature in features_to_diff:
        df[f'{feature}_Diff'] = df[f'T1_{feature}'] - df[f'T2_{feature}']
        
    return df

In [19]:
import itertools

def generate_2026_test_matrix(stats_df, season_to_predict=2026):
    """
    Generates every possible T1 vs T2 combination for a given season using itertools.
    Enforces the Kaggle standard where T1_TeamID < T2_TeamID.
    """
    # Filter stats to just the target season
    current_stats = stats_df[stats_df['Season'] == season_to_predict]
    
    # Separate Men's and Women's teams
    mens_teams = current_stats[current_stats['Is_Mens_Tourney'] == 1]['TeamID'].unique()
    womens_teams = current_stats[current_stats['Is_Mens_Tourney'] == 0]['TeamID'].unique()
    
    # Generate all pairwise combinations
    mens_combos = list(itertools.combinations(sorted(mens_teams), 2))
    womens_combos = list(itertools.combinations(sorted(womens_teams), 2))
    
    # Build DataFrames
    mens_df = pd.DataFrame(mens_combos, columns=['T1_TeamID', 'T2_TeamID'])
    mens_df['Season'] = season_to_predict
    mens_df['Is_Mens_Tourney'] = 1
    
    womens_df = pd.DataFrame(womens_combos, columns=['T1_TeamID', 'T2_TeamID'])
    womens_df['Season'] = season_to_predict
    womens_df['Is_Mens_Tourney'] = 0
    
    # Combine them
    test_matrix = pd.concat([mens_df, womens_df], ignore_index=True)
    
    # Generate the Kaggle ID format: Season_T1_T2
    test_matrix['ID'] = test_matrix['Season'].astype(str) + '_' + \
                        test_matrix['T1_TeamID'].astype(str) + '_' + \
                        test_matrix['T2_TeamID'].astype(str)
                        
    return test_matrix

# 1. Generate the raw combinations
raw_2026_matchups = generate_2026_test_matrix(master_stats, season_to_predict=2026)

# 2. Map the features using the EXACT same function we used for the training data
X_test_2026 = map_features_to_matchups(raw_2026_matchups, master_stats)

print(f"Generated {len(X_test_2026)} total matchups to predict for 2026.")
X_test_2026.head()

Generated 132133 total matchups to predict for 2026.


,T1_TeamID,T2_TeamID,Season,Is_Mens_Tourney,ID,T1_Avg_Off_Eff,T1_Avg_Def_Eff,T1_Net_Rating,T1_Win_Pct,T1_End_Season_Elo,T1_Overall_Win_Pct,T1_Home_Win_Pct,T1_Away_Win_Pct,T1_Avg_Score,T1_Avg_FGM,T1_Avg_FGA,T1_Avg_FGM3,T1_Avg_FGA3,T1_Avg_FTM,T1_Avg_FTA,T1_Avg_OR,T1_Avg_DR,T1_Avg_Ast,T1_Avg_TO,T1_Avg_Stl,T1_Avg_Blk,T1_Avg_PF,T1_Avg_Opp_Score,T1_Avg_Opp_FGM,T1_Avg_Opp_FGA,T1_Avg_Opp_FGM3,T1_Avg_Opp_FGA3,T1_Avg_Opp_FTM,T1_Avg_Opp_FTA,T1_Avg_Opp_OR,T1_Avg_Opp_DR,T1_Avg_Opp_Ast,T1_Avg_Opp_TO,T1_Avg_Opp_Stl,T1_Avg_Opp_Blk,T1_Avg_Opp_PF,T1_seed,T1_Last_14_Days_Win_Pct,T2_Avg_Off_Eff,T2_Avg_Def_Eff,T2_Net_Rating,T2_Win_Pct,T2_End_Season_Elo,T2_Overall_Win_Pct,T2_Home_Win_Pct,T2_Away_Win_Pct,T2_Avg_Score,T2_Avg_FGM,T2_Avg_FGA,T2_Avg_FGM3,T2_Avg_FGA3,T2_Avg_FTM,T2_Avg_FTA,T2_Avg_OR,T2_Avg_DR,T2_Avg_Ast,T2_Avg_TO,T2_Avg_Stl,T2_Avg_Blk,T2_Avg_PF,T2_Avg_Opp_Score,T2_Avg_Opp_FGM,T2_Avg_Opp_FGA,T2_Avg_Opp_FGM3,T2_Avg_Opp_FGA3,T2_Avg_Opp_FTM,T2_Avg_Opp_FTA,T2_Avg_Opp_OR,T2_Avg_Opp_DR,T2_Avg_Opp_Ast,T2_Avg_Opp_TO,T2_Avg_Opp_Stl,T2_Avg_Opp_Blk,T2_Avg_Opp_PF,T2_seed,T2_Last_14_Days_Win_Pct,Avg_Off_Eff_Diff,Avg_Def_Eff_Diff,Net_Rating_Diff,End_Season_Elo_Diff,Overall_Win_Pct_Diff,Home_Win_Pct_Diff,Away_Win_Pct_Diff,Avg_Score_Diff,Avg_FGM_Diff,Avg_FGA_Diff,Avg_FGM3_Diff,Avg_FGA3_Diff,Avg_FTM_Diff,Avg_FTA_Diff,Avg_OR_Diff,Avg_DR_Diff,Avg_Ast_Diff,Avg_TO_Diff,Avg_Stl_Diff,Avg_Blk_Diff,Avg_PF_Diff,Avg_Opp_Score_Diff,Avg_Opp_FGM_Diff,Avg_Opp_FGA_Diff,Avg_Opp_FGM3_Diff,Avg_Opp_FGA3_Diff,Avg_Opp_FTM_Diff,Avg_Opp_FTA_Diff,Avg_Opp_OR_Diff,Avg_Opp_DR_Diff,Avg_Opp_Ast_Diff,Avg_Opp_TO_Diff,Avg_Opp_Stl_Diff,Avg_Opp_Blk_Diff,Avg_Opp_PF_Diff,seed_Diff,Last_14_Days_Win_Pct_Diff
0,1101,1102,2026,1,2026_1101_1102,96.704899,104.901665,-8.196766,0.366667,1401.656257,0.366667,0.545455,0.2,67.703704,23.081481,54.403704,5.559259,17.862963,15.981481,22.674074,8.318519,18.203704,12.492593,13.466667,9.551852,2.562963,21.237037,73.625926,24.466667,48.581481,5.955556,14.555556,18.737037,25.937037,6.459259,21.911111,12.411111,15.396296,8.477778,4.607407,18.485185,17.0,0.25,90.677144,117.889518,-27.212374,0.093750,1168.510730,0.093750,0.166667,0.000000,61.531250,21.750000,51.156250,6.531250,21.375000,11.500000,17.937500,5.812500,20.531250,12.156250,13.437500,5.625000,2.375000,17.812500,79.875000,28.125000,57.312500,8.375000,22.250000,15.250000,20.968750,8.656250,24.250000,15.250000,9.812500,7.968750,3.531250,17.781250,17.0,0.000000,6.027755,-12.987853,19.015608,233.145528,0.272917,0.378788,0.200000,6.172454,1.331481,3.247454,-0.971991,-3.512037,4.481481,4.736574,2.506019,-2.327546,0.336343,0.029167,3.926852,0.187963,3.424537,-6.249074,-3.658333,-8.731019,-2.419444,-7.694444,3.487037,4.968287,-2.196991,-2.338889,-2.838889,5.583796,0.509028,1.076157,0.703935,0.0,0.250000
1,1101,1103,2026,1,2026_1101_1103,96.704899,104.901665,-8.196766,0.366667,1401.656257,0.366667,0.545455,0.2,67.703704,23.081481,54.403704,5.559259,17.862963,15.981481,22.674074,8.318519,18.203704,12.492593,13.466667,9.551852,2.562963,21.237037,73.625926,24.466667,48.581481,5.955556,14.555556,18.737037,25.937037,6.459259,21.911111,12.411111,15.396296,8.477778,4.607407,18.485185,17.0,0.25,120.218619,103.139087,17.079531,0.843750,1790.463108,0.843750,1.000000,0.750000,86.875000,31.156250,62.593750,10.875000,28.250000,13.687500,18.187500,9.562500,24.656250,17.875000,10.531250,7.250000,2.875000,16.281250,74.500000,26.031250,60.562500,9.187500,26.187500,13.250000,18.343750,9.375000,19.718750,13.218750,12.375000,6.281250,2.843750,16.093750,12.0,1.000000,-23.513720,1.762578,-25.276297,-388.806850,-0.477083,-0.454545,-0.550000,-19.171296,-8.074769,-8.190046,-5.315741,-10.387037,2.293981,4.486574,-1.243981,-6.452546,-5.382407,2.935417,2.301852,-0.312037,4.955787,-0.874074,-1.564583,-11.981019,-3.231944,-11.631944,5.487037,7.593287,-2.915741,2.192361,-0.807639,3.021296,2.196528,1.763657,2.391435,5.0,-0.750000
2,1101,1104,2026,1,2026_1101_1104,96.704899,104.901665,-8.196766,0.366667,1401.656257,0.366667,0.545455,0.2,67.7037

In [20]:
# 1. Map your End-of-Season Master Stats onto the massive Regular Season dataset
train_dataset = map_features_to_matchups(regular_data, master_stats)

# 2. Drop all "In-Game" stats to prevent Data Leakage
# We also drop 'T1_Loc' because the Kaggle 2026 Test Set doesn't provide locations, 
# so the model can't rely on it for predictions!
cols_to_drop = [
    'DayNum', 'T1_Score', 'T2_Score', 'T1_Loc',
    'T1_FGM', 'T2_FGM', 'T1_FGA', 'T2_FGA',
    'T1_FGM3', 'T2_FGM3', 'T1_FGA3', 'T2_FGA3',
    'T1_FTM', 'T2_FTM', 'T1_FTA', 'T2_FTA',
    'T1_OR', 'T2_OR', 'T1_DR', 'T2_DR',
    'T1_Ast', 'T2_Ast', 'T1_TO', 'T2_TO',
    'T1_Stl', 'T2_Stl', 'T1_Blk', 'T2_Blk',
    'T1_PF', 'T2_PF'
]

# Clean the dataset
train_dataset = train_dataset.drop(columns=[c for c in cols_to_drop if c in train_dataset.columns])

# 3. Split into Features (X) and Target (y)
X_train = train_dataset.drop(columns=['Target_Win'])
y_train = train_dataset['Target_Win']

print(f"🚀 Generalist Model Training on {len(X_train)} historical games!")
X_train

🚀 Generalist Model Training on 423432 historical games!


,Season,T1_TeamID,T2_TeamID,Is_Mens_Tourney,T1_Avg_Off_Eff,T1_Avg_Def_Eff,T1_Net_Rating,T1_Win_Pct,T1_End_Season_Elo,T1_Overall_Win_Pct,T1_Home_Win_Pct,T1_Away_Win_Pct,T1_Avg_Score,T1_Avg_FGM,T1_Avg_FGA,T1_Avg_FGM3,T1_Avg_FGA3,T1_Avg_FTM,T1_Avg_FTA,T1_Avg_OR,T1_Avg_DR,T1_Avg_Ast,T1_Avg_TO,T1_Avg_Stl,T1_Avg_Blk,T1_Avg_PF,T1_Avg_Opp_Score,T1_Avg_Opp_FGM,T1_Avg_Opp_FGA,T1_Avg_Opp_FGM3,T1_Avg_Opp_FGA3,T1_Avg_Opp_FTM,T1_Avg_Opp_FTA,T1_Avg_Opp_OR,T1_Avg_Opp_DR,T1_Avg_Opp_Ast,T1_Avg_Opp_TO,T1_Avg_Opp_Stl,T1_Avg_Opp_Blk,T1_Avg_Opp_PF,T1_seed,T1_Last_14_Days_Win_Pct,T2_Avg_Off_Eff,T2_Avg_Def_Eff,T2_Net_Rating,T2_Win_Pct,T2_End_Season_Elo,T2_Overall_Win_Pct,T2_Home_Win_Pct,T2_Away_Win_Pct,T2_Avg_Score,T2_Avg_FGM,T2_Avg_FGA,T2_Avg_FGM3,T2_Avg_FGA3,T2_Avg_FTM,T2_Avg_FTA,T2_Avg_OR,T2_Avg_DR,T2_Avg_Ast,T2_Avg_TO,T2_Avg_Stl,T2_Avg_Blk,T2_Avg_PF,T2_Avg_Opp_Score,T2_Avg_Opp_FGM,T2_Avg_Opp_FGA,T2_Avg_Opp_FGM3,T2_Avg_Opp_FGA3,T2_Avg_Opp_FTM,T2_Avg_Opp_FTA,T2_Avg_Opp_OR,T2_Avg_Opp_DR,T2_Avg_Opp_Ast,T2_Avg_Opp_TO,T2_Avg_Opp_Stl,T2_Avg_Opp_Blk,T2_Avg_Opp_PF,T2_seed,T2_Last_14_Days_Win_Pct,Avg_Off_Eff_Diff,Avg_Def_Eff_Diff,Net_Rating_Diff,End_Season_Elo_Diff,Overall_Win_Pct_Diff,Home_Win_Pct_Diff,Away_Win_Pct_Diff,Avg_Score_Diff,Avg_FGM_Diff,Avg_FGA_Diff,Avg_FGM3_Diff,Avg_FGA3_Diff,Avg_FTM_Diff,Avg_FTA_Diff,Avg_OR_Diff,Avg_DR_Diff,Avg_Ast_Diff,Avg_TO_Diff,Avg_Stl_Diff,Avg_Blk_Diff,Avg_PF_Diff,Avg_Opp_Score_Diff,Avg_Opp_FGM_Diff,Avg_Opp_FGA_Diff,Avg_Opp_FGM3_Diff,Avg_Opp_FGA3_Diff,Avg_Opp_FTM_Diff,Avg_Opp_FTA_Diff,Avg_Opp_OR_Diff,Avg_Opp_DR_Diff,Avg_Opp_Ast_Diff,Avg_Opp_TO_Diff,Avg_Opp_Stl_Diff,Avg_Opp_Blk_Diff,Avg_Opp_PF_Diff,seed_Diff,Last_14_Days_Win_Pct_Diff
0,2003,1104,1328,1,103.557697,97.794103,5.763594,0.607143,1571.793690,0.607143,0.866667,0.111111,69.015873,23.940476,56.964286,6.329365,19.805556,14.805556,20.849206,13.523810,23.853175,12.059524,13.234127,6.571429,3.777778,17.972222,64.753968,23.150794,55.297619,6.345238,19.067460,12.107143,17.095238,10.857143,22.551587,11.638889,13.789683,5.503968,3.170635,19.170635,10.0,0.333333,108.524206,92.240245,16.283961,0.800000,1765.072816,0.800000,0.937500,0.625000,70.325926,24.966667,55.922222,7.388889,18.781481,13.003704,18.344444,12.003704,24.696296,14.014815,11.659259,6.870370,3.725926,18.366667,59.377778,21.303704,52.707407,4.411111,13.477778,12.359259,18.385185,10.266667,22.170370,10.329630,13.588889,5.122222,2.629630,17.314815,1.0,0.80,-4.966509,5.553858,-10.520367,-193.279126,-0.192857,-0.070833,-0.513889,-1.310053,-1.026190,1.042063,-1.059524,1.024074,1.801852,2.504762,1.520106,-0.843122,-1.955291,1.574868,-0.298942,0.051852,-0.394444,5.376190,1.847090,2.590212,1.934127,5.589683,-0.252116,-1.289947,0.590476,0.381217,1.309259,0.200794,0.381746,0.541005,1.855820,9.0,-0.466667
1,2003,1272,1393,1,105.639144,93.245313,12.393831,0.793103,1746.960988,0.793103,0.875000,0.636364,74.210728,26.183908,59.762452,6.969349,19.996169,14.873563,22.766284,14.019157,25.846743,16.555556,13.739464,7.363985,5.042146,18.693487,65.517241,23.164751,57.593870,5.854406,18.218391,13.333333,20.659004,12.295019,23.482759,13.237548,15.019157,7.252874,3.153257,19.827586,7.0,0.750000,110.159529,96.446601,13.712927,0.827586,1724.703180,0.827586,1.000000,0.666667,79.747126,29.126437,61.908046,5.222222,15.781609,16.272031,23.463602,14.218391,26.758621,14.938697,13.570881,8.291188,7.233716,16.486590,69.574713,25.107280,64.344828,6.908046,22.528736,12.452107,18.766284,15.743295,22.160920,15.819923,14.409962,6.601533,3.099617,19.605364,3.0,0.75,-4.520384,-3.201288,-1.319096,22.257808,-0.034483,-0.125000,-0.030303,-5.536398,-2.942529,-2.145594,1.747126,4.214559,-1.398467,-0.697318,-0.199234,-0.911877,1.616858,0.168582,-0.927203,-2.191571,2.206897,-4.057471,-1.942529,-6.750958,-1.053640,-4.310345,0.881226,1.892720,-3.448276,1.321839,-2.582375,0.609195,0.651341,0.053640,0.222222,4.0,0.000000
2,2003,1266,1437,1,115.464640,99.308874,16.155766,0.821429,1750.077905,0.821429,0.937500,0.700000,78.055556,27.079365,55.952381,5.765873,15.1

In [28]:
import xgboost as xgb
import pandas as pd

# 1. Define the 35 best features (Focusing on Differentials, Elo, and Efficiency)
best_35_features = [
    # Top Tier Power Metrics
    'End_Season_Elo_Diff', 
    'Net_Rating_Diff', 
    'seed_Diff', 
    'Overall_Win_Pct_Diff', 
    'Last_14_Days_Win_Pct_Diff',
    'Avg_Off_Eff_Diff', 
    'Avg_Def_Eff_Diff',
    
    # Core Offensive Stat Differentials
    'Avg_Score_Diff', 'Avg_FGM_Diff', 'Avg_FGA_Diff', 
    'Avg_FGM3_Diff', 'Avg_FGA3_Diff', 'Avg_FTM_Diff', 
    'Avg_FTA_Diff', 'Avg_OR_Diff', 'Avg_DR_Diff', 
    'Avg_Ast_Diff', 'Avg_TO_Diff', 'Avg_Stl_Diff', 
    'Avg_Blk_Diff', 'Avg_PF_Diff',
    
    # Core Defensive (Opponent) Stat Differentials
    'Avg_Opp_Score_Diff', 'Avg_Opp_FGM_Diff', 'Avg_Opp_FGA_Diff', 
    'Avg_Opp_FGM3_Diff', 'Avg_Opp_FGA3_Diff', 'Avg_Opp_FTM_Diff', 
    'Avg_Opp_FTA_Diff', 'Avg_Opp_OR_Diff', 'Avg_Opp_DR_Diff', 
    'Avg_Opp_Ast_Diff', 'Avg_Opp_TO_Diff', 'Avg_Opp_Stl_Diff', 
    'Avg_Opp_Blk_Diff', 'Avg_Opp_PF_Diff'
]

# 2. Filter the training and testing sets to just these 35 features
X_train_xgb = X_train[best_35_features]
X_test_xgb = X_test_2026[best_35_features]

print("Training XGBoost model on 35 features... hang tight!")

# 3. Initialize and train the XGBoost Classifier
# Using parameters that balance speed and preventing overfitting
xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1 # Uses all your CPU cores to run faster
)

xgb_model.fit(X_train_xgb, y_train)

# 4. Predict probabilities for the 2026 Test Set
# We want the probability of Class 1 (Target_Win == 1)
print("Predicting 2026 matchups...")
test_preds = xgb_model.predict_proba(X_test_xgb)[:, 1]

# 5. Format and save the submission
submission = pd.DataFrame({
    'ID': X_test_2026['ID'],
    'Pred': test_preds
})

submission.to_csv('submission.csv', index=False)
print("Done! 🎉 submission.csv has been saved and is ready to upload to Kaggle.")

Training XGBoost model on 35 features... hang tight!
Predicting 2026 matchups...
Done! 🎉 submission.csv has been saved and is ready to upload to Kaggle.


In [29]:
import numpy as np
from sklearn.metrics import brier_score_loss
import xgboost as xgb

print("Starting Leave-One-Season-Out Validation...")
print("Grab a quick glass of water, training on ~400k rows 20+ times will take a few minutes! ⏱️")

# Get the list of all unique seasons in the training data
seasons = sorted(train_dataset['Season'].unique())
brier_scores = []

for val_season in seasons:
    # 1. Split data: Hold out one season, train on everything else
    train_split = train_dataset[train_dataset['Season'] != val_season]
    val_split = train_dataset[train_dataset['Season'] == val_season]
    
    X_train_cv = train_split[best_35_features]
    y_train_cv = train_split['Target_Win']
    
    X_val_cv = val_split[best_35_features]
    y_val_cv = val_split['Target_Win']
    
    # 2. Initialize model (Using the exact same parameters as your main model)
    xgb_cv = xgb.XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='logloss', 
        random_state=42,
        n_jobs=-1 
    )
    
    # 3. Fit the model
    xgb_cv.fit(X_train_cv, y_train_cv)
    
    # 4. Predict probabilities for the holdout season
    val_preds = xgb_cv.predict_proba(X_val_cv)[:, 1]
    
    # 5. Calculate Brier Score
    brier = brier_score_loss(y_val_cv, val_preds)
    brier_scores.append(brier)
    
    print(f"Season {val_season} Brier Score: {brier:.5f}")

# 6. Calculate the average across all seasons
mean_brier = np.mean(brier_scores)
print("-" * 30)
print(f"🌟 Rough Overall Brier Score: {mean_brier:.5f} 🌟")

Starting Leave-One-Season-Out Validation...
Grab a quick glass of water, training on ~400k rows 20+ times will take a few minutes! ⏱️
Season 2003 Brier Score: 0.17173
Season 2004 Brier Score: 0.16624
Season 2005 Brier Score: 0.17073
Season 2006 Brier Score: 0.16919
Season 2007 Brier Score: 0.16898
Season 2008 Brier Score: 0.16915
Season 2009 Brier Score: 0.17010
Season 2010 Brier Score: 0.15913
Season 2011 Brier Score: 0.15890
Season 2012 Brier Score: 0.15657
Season 2013 Brier Score: 0.16116
Season 2014 Brier Score: 0.16001
Season 2015 Brier Score: 0.16096
Season 2016 Brier Score: 0.15777
Season 2017 Brier Score: 0.15999
Season 2018 Brier Score: 0.15894
Season 2019 Brier Score: 0.15910
Season 2020 Brier Score: 0.16568
Season 2021 Brier Score: 0.16190
Season 2022 Brier Score: 0.15805
Season 2023 Brier Score: 0.16360
Season 2024 Brier Score: 0.16332
Season 2025 Brier Score: 0.15750
Season 2026 Brier Score: 0.15766
------------------------------
🌟 Rough Overall Brier Score: 0.16277 🌟
Good